In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RetailFeatures") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.1.1


In [5]:
# Start a local Spark session
spark = SparkSession.builder \
    .appName("RetailFeatures") \
    .master("local[*]") \
    .getOrCreate()

In [7]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/azeemkhalipha/mlops-demand-forecast/.env")

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

PROJECT_ROOT = os.getenv("PROJECT_ROOT")
RAW_DATA_PATH = f"{PROJECT_ROOT}/data/raw/online_retail_II.csv"
FEATURES_PATH = f"{PROJECT_ROOT}/data/features"

# print(f"Project root: {PROJECT_ROOT}")
print(f"Data exists: {os.path.exists(RAW_DATA_PATH)}")

python-dotenv could not parse statement starting at line 1


Data exists: True


In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("RetailFeatures") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.csv(
    RAW_DATA_PATH,
    header=True,
    inferSchema=True
)

df.printSchema()
print(f"Total rows: {df.count():,}")

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)

Total rows: 1,067,371


In [9]:
# Explore the data
df.printSchema()
df.show(5)
print(f"Total rows: {df.count():,}")

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|"RECORD FRAME 7""...|      4

In [10]:
# Data transformations
df = df.withColumnRenamed("Customer ID", "customer_id")
df = df.withColumn("total_revenue", F.col("Quantity") * F.col("Price"))
df = df.filter(F.col("Quantity") > 0)
df = df.filter(~F.col("Invoice").startswith("C"))
df = df.dropna(subset=["customer_id"])

In [11]:
df.groupBy("Country") \
  .agg(F.sum("total_revenue").alias("revenue")) \
  .orderBy("revenue", ascending=False) \
  .show(10)

+--------------+--------------------+
|       Country|             revenue|
+--------------+--------------------+
|United Kingdom|1.4723147516997937E7|
|          EIRE|   621631.1099999981|
|   Netherlands|   554232.3399999995|
|       Germany|   431262.4610000004|
|        France|  355257.47000000015|
|     Australia|  169968.10999999993|
|         Spain|  109178.53000000009|
|   Switzerland|   100365.3400000001|
|        Sweden|   91549.71999999999|
|       Denmark|            69862.19|
+--------------+--------------------+
only showing top 10 rows


In [12]:
FEATURES_PATH = f"{PROJECT_ROOT}/data/features"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("DateFeatures") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark ready")

Spark ready


26/05/14 23:07:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [13]:
df = spark.read.csv(RAW_DATA_PATH, header=True, inferSchema=True)
df = df.withColumnRenamed("Customer ID", "customer_id")
df = df.filter(F.col("Quantity") > 0)
df = df.filter(F.col("Price") > 0)
df = df.filter(~F.col("Invoice").startswith("C"))
df = df.dropna(subset=["customer_id"])
print(f"Clean rows: {df.count():,}")

Clean rows: 805,549


In [14]:
def add_date_features(df):
    df = df.withColumn("invoice_date", F.to_date(F.col("InvoiceDate")))
    df = df.withColumn("year",         F.year("invoice_date"))
    df = df.withColumn("month",        F.month("invoice_date"))
    df = df.withColumn("day",          F.dayofmonth("invoice_date"))
    df = df.withColumn("dayofweek",    F.dayofweek("invoice_date"))
    df = df.withColumn("weekofyear",   F.weekofyear("invoice_date"))
    df = df.withColumn("quarter",      F.quarter("invoice_date"))
    df = df.withColumn("is_weekend",
        F.when(F.col("dayofweek").isin([1, 7]), 1).otherwise(0))
    df = df.withColumn("is_month_end",
        F.when(F.col("day") >= 25, 1).otherwise(0))
    return df

df = add_date_features(df)
df.select("invoice_date","year","month","dayofweek","is_weekend","is_month_end").show(10)

+------------+----+-----+---------+----------+------------+
|invoice_date|year|month|dayofweek|is_weekend|is_month_end|
+------------+----+-----+---------+----------+------------+
|  2009-12-01|2009|   12|        3|         0|           0|
|  2009-12-01|2009|   12|        3|         0|           0|
|  2009-12-01|2009|   12|        3|         0|           0|
|  2009-12-01|2009|   12|        3|         0|           0|
|  2009-12-01|2009|   12|        3|         0|           0|
|  2009-12-01|2009|   12|        3|         0|           0|
|  2009-12-01|2009|   12|        3|         0|           0|
|  2009-12-01|2009|   12|        3|         0|           0|
|  2009-12-01|2009|   12|        3|         0|           0|
|  2009-12-01|2009|   12|        3|         0|           0|
+------------+----+-----+---------+----------+------------+
only showing top 10 rows


Lag and rolling features

In [15]:
from pyspark.sql.window import Window

def add_lag_features(df):
    # Aggregate to daily level per product
    daily = df.groupBy("StockCode", "invoice_date").agg(
        F.sum("Quantity").alias("daily_qty"),
        F.sum(F.col("Quantity") * F.col("Price")).alias("daily_revenue")
    )

    # Window ordered by date within each product
    w = Window.partitionBy("StockCode").orderBy("invoice_date")

    # Lag features
    daily = daily.withColumn("qty_lag_1",  F.lag("daily_qty", 1).over(w))
    daily = daily.withColumn("qty_lag_7",  F.lag("daily_qty", 7).over(w))
    daily = daily.withColumn("qty_lag_30", F.lag("daily_qty", 30).over(w))

    # Rolling averages
    w7  = Window.partitionBy("StockCode").orderBy("invoice_date").rowsBetween(-7, -1)
    w30 = Window.partitionBy("StockCode").orderBy("invoice_date").rowsBetween(-30, -1)

    daily = daily.withColumn("qty_rolling_avg_7",  F.avg("daily_qty").over(w7))
    daily = daily.withColumn("qty_rolling_avg_30", F.avg("daily_qty").over(w30))
    daily = daily.withColumn("qty_rolling_std_7",  F.stddev("daily_qty").over(w7))

    # Drop rows where lag features are null
    daily = daily.dropna(subset=["qty_lag_30"])
    return daily

lag_df = add_lag_features(df)
lag_df.show(5)
print(f"Lag feature rows: {lag_df.count():,}")
print(f"Columns: {lag_df.columns}")

+---------+------------+---------+-------------+---------+---------+----------+------------------+------------------+------------------+
|StockCode|invoice_date|daily_qty|daily_revenue|qty_lag_1|qty_lag_7|qty_lag_30| qty_rolling_avg_7|qty_rolling_avg_30| qty_rolling_std_7|
+---------+------------+---------+-------------+---------+---------+----------+------------------+------------------+------------------+
|    10133|  2010-04-28|       10|          8.5|       10|        5|         6| 5.714285714285714|15.866666666666667| 4.231401884099631|
|    10133|  2010-04-29|        4|          3.4|       10|        1|        40| 6.428571428571429|              16.0| 4.503966505838414|
|    10133|  2010-05-10|       30|         25.5|        4|        3|        24| 6.857142857142857|              14.8|4.0178174601214955|
|    10133|  2010-05-11|       20|         17.0|       30|       10|        10|10.714285714285714|              15.0| 9.250482612892613|
|    10133|  2010-05-13|       10|       

Lag feature rows: 330,546
Columns: ['StockCode', 'invoice_date', 'daily_qty', 'daily_revenue', 'qty_lag_1', 'qty_lag_7', 'qty_lag_30', 'qty_rolling_avg_7', 'qty_rolling_avg_30', 'qty_rolling_std_7']


Customer features

In [16]:
def add_customer_features(df):
    customer_stats = df.groupBy("customer_id").agg(
        F.sum(F.col("Quantity") * F.col("Price")).alias("customer_total_spend"),
        F.countDistinct("Invoice").alias("customer_total_orders"),
        F.avg(F.col("Quantity") * F.col("Price")).alias("customer_avg_order_value"),
        F.countDistinct("StockCode").alias("customer_unique_products"),
        F.min("invoice_date").alias("customer_first_purchase"),
        F.max("invoice_date").alias("customer_last_purchase")
    )

    customer_stats = customer_stats.withColumn(
        "customer_lifetime_days",
        F.datediff(
            F.col("customer_last_purchase"),
            F.col("customer_first_purchase")
        )
    )

    customer_stats = customer_stats.withColumn(
        "customer_segment",
        F.when(F.col("customer_total_spend") > 5000, "high_value")
         .when(F.col("customer_total_spend") > 1000, "mid_value")
         .otherwise("low_value")
    )

    df = df.join(customer_stats, on="customer_id", how="left")
    return df, customer_stats

df, customer_stats = add_customer_features(df)
customer_stats.show(5)
print(f"Unique customers: {customer_stats.count():,}")

+-----------+--------------------+---------------------+------------------------+------------------------+-----------------------+----------------------+----------------------+----------------+
|customer_id|customer_total_spend|customer_total_orders|customer_avg_order_value|customer_unique_products|customer_first_purchase|customer_last_purchase|customer_lifetime_days|customer_segment|
+-----------+--------------------+---------------------+------------------------+------------------------+-----------------------+----------------------+----------------------+----------------+
|    17072.0|              282.05|                    1|      12.820454545454545|                      20|             2010-03-24|            2010-03-24|                     0|       low_value|
|    17884.0|  3072.8900000000003|                   20|       6.375290456431536|                     305|             2009-12-04|            2011-12-06|                   732|       mid_value|
|    16822.0|              181

In [17]:
import os

# Create output folders
os.makedirs(f"{FEATURES_PATH}/ml_features", exist_ok=True)
os.makedirs(f"{FEATURES_PATH}/customer_features", exist_ok=True)

# Save lag features
lag_df.write.mode("overwrite").parquet(
    f"{FEATURES_PATH}/ml_features"
)

# Save customer features
customer_stats.write.mode("overwrite").parquet(
    f"{FEATURES_PATH}/customer_features"
)

print("Feature datasets saved successfully.")
print(f"ML features: {lag_df.count():,} rows, {len(lag_df.columns)} columns")
print(f"Customer features: {customer_stats.count():,} rows")

Feature datasets saved successfully.


ML features: 330,546 rows, 10 columns
Customer features: 5,878 rows


In [18]:
import os

ml_files = os.listdir(f"{FEATURES_PATH}/ml_features")
cust_files = os.listdir(f"{FEATURES_PATH}/customer_features")

print("ML feature files:", ml_files)
print("Customer feature files:", cust_files)

ML feature files: ['part-00005-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet', '.part-00005-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet.crc', 'part-00002-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet', '._SUCCESS.crc', '.part-00002-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet.crc', 'part-00003-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet', 'part-00004-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet', '.part-00001-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet.crc', 'part-00001-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet', '.part-00004-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet.crc', '_SUCCESS', '.part-00003-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet.crc', 'part-00000-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet', '.part-00000-f42338e6-4616-462d-993b-998eced4bc10-c000.snappy.parquet.crc']
Customer feature files: ['._SUCCESS.crc', '.part-00000-4e594aef-db0b-4